In [ ]:
# 1. Upload your Kaggle API token (kaggle.json)
from google.colab import files
print("▶ Upload your kaggle.json (from your Kaggle account settings)")
files.upload()   # select your kaggle.json

# 2. Configure Kaggle CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download & unzip the dataset
!mkdir -p data
!kaggle datasets download -d sunnysai12345/news-summary -p data/
!unzip -o data/news-summary.zip   -d data/

▶ Upload your kaggle.json (from your Kaggle account settings)


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sunnysai12345/news-summary
License(s): GPL-2.0
Archive:  data/news-summary.zip
  inflating: data/news_summary.csv   
  inflating: data/news_summary_more.csv  


In [ ]:
!pip install evaluate transformers datasets peft rouge bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ─── 1) Imports ─────────────────────────────────────────────────────────
import torch, numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, pipeline
)
from peft import LoraConfig, get_peft_model
from evaluate import load


In [ ]:
ds = load_dataset("csv", data_files="data/news_summary.csv", encoding="latin-1")["train"]
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, test_ds = split["train"], split["test"]

Map:   0%|          | 0/4062 [00:00<?, ? examples/s]

Map:   0%|          | 0/452 [00:00<?, ? examples/s]

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
MODEL_ID = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto"
)
base_model.config.use_cache = False  # disable KV cache

tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [ ]:
peft_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["query_key_value","dense_4h_to_h"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, peft_cfg)
model.config.use_cache = False      # ensure still no cache

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"🔧 Trainable params: {trainable:,}/{total:,} (~{100*trainable/total:.2f}%)")

🔧 Trainable params: 16,351,232/3,625,096,064 (~0.45%)


In [ ]:
def preprocess(batch):
    inputs  = ["Summarize: "+t for t in batch["text"]]
    targets = batch["headlines"]
    enc = tokenizer(inputs,  max_length=512, truncation=True, padding="longest")
    dec = tokenizer(targets, max_length=64,  truncation=True, padding="longest")
    input_ids, labels = [], []
    for i_ids, t_ids in zip(enc["input_ids"], dec["input_ids"]):
        input_ids.append(i_ids + t_ids[1:])
        labels.append([-100]*len(i_ids) + t_ids[1:])
    return {"input_ids":input_ids, "labels":labels}

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
test_ds  = test_ds.map(preprocess,  batched=True, remove_columns=test_ds.column_names)

Map:   0%|          | 0/4062 [00:00<?, ? examples/s]

Map:   0%|          | 0/452 [00:00<?, ? examples/s]

In [ ]:
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
args = TrainingArguments(
    output_dir="falcon-7b-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-4,
    fp16=True,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    tokenizer=tokenizer
)


<ipython-input-8-9d1d407b275d>:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amrwael-elwakil (amrwael-elwakil-msa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,2.286400
100,2.134700
150,2.054700
200,2.020000
250,1.991500
300,1.988500
350,1.970700
400,1.951600
450,1.964000
500,1.968500


TrainOutput(global_step=1521, training_loss=1.9175976935693264, metrics={'train_runtime': 3104.8421, 'train_samples_per_second': 3.925, 'train_steps_per_second': 0.49, 'total_flos': 7.025365600192512e+16, 'train_loss': 1.9175976935693264, 'epoch': 2.994583948793698})

In [ ]:
gen = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    # NO `device=` here
)

Device set to use cuda:0
The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FalconMambaForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'Gemma2ForCausalLM', 'Gemma3ForConditionalGeneration', 'Gemma3ForCausalLM', 'GitForCausalLM', 'GlmForCausalLM', 'Glm4ForCausalLM', 'GotOcr2ForConditionalGeneration', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoFo

In [ ]:
test_texts = raw_test["text"]
test_refs  = raw_test["headlines"]


In [32]:
outs = gen(
    ["Summarize: " + t for t in test_texts],
    max_new_tokens=64,
    num_beams=4,
    length_penalty=0.6,
    no_repeat_ngram_size=2
)
preds = [o[0]["generated_text"].strip() for o in outs]


In [39]:
!pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.5 MB/s eta 0:00:00


In [40]:
rouge     = load("rouge")
bertscore = load("bertscore")
bleu      = load("bleu")

In [45]:
r  = rouge.compute(predictions=preds, references=test_refs)
rs = {k: v*100 for k,v in r.items()}
bf1 = np.mean(bertscore.compute(
    predictions=preds,
    references=test_refs,
    model_type="roberta-large"
)["f1"])*100
bl  = bleu.compute(predictions=preds, references=[[r] for r in test_refs])["bleu"]*100

In [47]:
print(f"\n📊 ROUGE-1: {rs['rouge1']:.2f}%, ROUGE-2: {rs['rouge2']:.2f}%, ROUGE-L: {rs['rougeL']:.2f}%")
print(f"    BERTScore-F1: {bf1:.2f}%, BLEU: {bl:.2f}%")



📊 ROUGE-1: 16.05%, ROUGE-2: 7.31%, ROUGE-L: 13.49%
    BERTScore-F1: 85.57%, BLEU: 1.94%
